In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


--2026-10-12 22:18:32--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.004s  

2026-10-12 22:18:32 (71.3 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [6]:
import pandas as pd
df = pd.read_csv('course_lead_scoring_2026.csv')

In [7]:
df = df.fillna({col: 'NA' if df[col].dtype == 'object' else 0.0 for col in df.columns})

# Q.1)

In [8]:
# Get the most frequent observation (the mode)
industry_mode = df['industry'].mode()[0]
print(f"The mode for the 'industry' column is: {industry_mode}")

# Alternatively, use value_counts() to see the exact count for 'NA', 'technology', 'healthcare', and 'retail'
print(df['industry'].value_counts())

The mode for the 'industry' column is: technology
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
0.0               240
Name: count, dtype: int64


# Q.2)

In [9]:
# Select only numerical columns for the correlation matrix
numerical_df = df.select_dtypes(exclude=['object'])

# Create the correlation matrix
corr_matrix = numerical_df.corr()

# Check the correlation coefficients for the specific pairs
print("interaction_count & lead_score:", corr_matrix.loc['interaction_count', 'lead_score'])
print("number_of_courses_viewed & lead_score:", corr_matrix.loc['number_of_courses_viewed', 'lead_score'])
print("number_of_courses_viewed & interaction_count:", corr_matrix.loc['number_of_courses_viewed', 'interaction_count'])
print("annual_income & interaction_count:", corr_matrix.loc['annual_income', 'interaction_count'])

interaction_count & lead_score: 0.9157457980418697
number_of_courses_viewed & lead_score: 0.7572042392104886
number_of_courses_viewed & interaction_count: 0.7216090038937035
annual_income & interaction_count: 0.12284235135955959


In [10]:
# answer is interaction_count & lead_score

In [11]:
from sklearn.model_selection import train_test_split

# Split the data with the exact calls provided
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Extract the target values
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Make sure that the target value 'converted' is not in your dataframe
del df_train['converted']
del df_val['converted']
del df_test['converted']


# Q.3)

In [14]:
from sklearn.metrics import mutual_info_score

categorical_cols = ['industry', 'location', 'lead_source', 'employment_status']

for col in categorical_cols:
    # .astype(str) forces all values in the column to be strings
    score = mutual_info_score(df_train[col].astype(str), y_train)
    print(f"{col}: {round(score, 2)}")

industry: 0.0
location: 0.0
lead_source: 0.03
employment_status: 0.02


In [15]:
# Answer is lead_source

# Q.4)

In [16]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. One-hot encode the categorical variables
dv = DictVectorizer(sparse=False)

# Convert dataframes to dictionaries
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

# Fit the vectorizer on the training data and transform both sets
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# 2. Fit the logistic regression model on the training dataset
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 3. Calculate the accuracy on the validation dataset
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {round(accuracy, 2)}")

Validation Accuracy: 0.65


In [17]:
# answer is 0.65

# Q.5)

In [18]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Calculate the original unrounded accuracy
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
original_accuracy = accuracy_score(y_val, model.predict(X_val))

print(f"Original Accuracy: {original_accuracy}\n")

# 2. Define the features to eliminate
features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

# 3. Exclude each feature, train a new model, and calculate the difference
for feature in features_to_check:
    # Drop the feature from training and validation sets
    df_train_reduced = df_train.drop(columns=[feature])
    df_val_reduced = df_val.drop(columns=[feature])
    
    # One-hot encode the reduced datasets
    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced = dv_reduced.fit_transform(df_train_reduced.to_dict(orient='records'))
    X_val_reduced = dv_reduced.transform(df_val_reduced.to_dict(orient='records'))
    
    # Train the model with the exact same parameters
    model_reduced = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_reduced.fit(X_train_reduced, y_train)
    
    # Calculate new accuracy and the difference
    new_accuracy = accuracy_score(y_val, model_reduced.predict(X_val_reduced))
    difference = original_accuracy - new_accuracy
    
    print(f"Excluded '{feature}' -> Difference: {difference}")

Original Accuracy: 0.645

Excluded 'lead_source' -> Difference: 0.0030000000000000027
Excluded 'number_of_courses_viewed' -> Difference: 0.0020000000000000018
Excluded 'interaction_count' -> Difference: 0.04400000000000004
